# M4 — Cellphone catalog prices and specifications

This exploration reads the M3 layer without changing it. Catalog prices are **not verified launch MSRP**. Android is an OS grouping, not a validated smartphone-only device class. See the generated [EDA report](../docs/eda_report.md).


In [1]:
# All analytical choices and seeds are set before any query or summary.
from pathlib import Path
import os, sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
PARAMS = {
    'brands': ['Samsung', 'Xiaomi', 'Huawei', 'Sony'],
    'os': 'Android', 'currency': 'EUR',
    'start_year': 2010, 'end_year': 2025,
    'flags': ['ok', 'unit_converted', 'unusual_low', 'unusual_high'],
    'strict_flags': ['ok', 'unit_converted'],
    'min_models': 20, 'min_prices': 10, 'min_coverage': 0.5,
    'min_joint': 10,
    'large_display_in': 7.0,
    'bin_trials': [50, 100, 200], 'bin_width': 100,
    'comparison': ['Samsung', 'Xiaomi'], 'comparison_start': 2015,
    'bootstrap_reps': 5000, 'block_length': 2, 'seed': 20261111,
}
DATABASE = Path(os.environ.get('PHONE_M4_DB', str(ROOT / 'data/phones.db')))
print(json.dumps(PARAMS, indent=2))
print('Database:', DATABASE.relative_to(ROOT).as_posix() if DATABASE.is_relative_to(ROOT) else DATABASE)


{
  "brands": [
    "Samsung",
    "Xiaomi",
    "Huawei",
    "Sony"
  ],
  "os": "Android",
  "currency": "EUR",
  "start_year": 2010,
  "end_year": 2025,
  "flags": [
    "ok",
    "unit_converted",
    "unusual_low",
    "unusual_high"
  ],
  "strict_flags": [
    "ok",
    "unit_converted"
  ],
  "min_models": 20,
  "min_prices": 10,
  "min_coverage": 0.5,
  "min_joint": 10,
  "large_display_in": 7.0,
  "bin_trials": [
    50,
    100,
    200
  ],
  "bin_width": 100,
  "comparison": [
    "Samsung",
    "Xiaomi"
  ],
  "comparison_start": 2015,
  "bootstrap_reps": 5000,
  "block_length": 2,
  "seed": 20261111
}
Database: data/phones.db


## Read-only build and analytical grain

Question: which observations can answer a price/specification question? The SQL pivots current fields to one row per source catalog ID; all joins keep the full latest-snapshot key. Numeric flag lists are bound parameters. OS must be mapped and year must be `ok`. The pipeline stops on repeated analytical fields or new identity mappings rather than silently collapsing them. The relationship scatter is written before coefficients are calculated. Figures use a standard-library SVG backend; there are no network or plotting-package dependencies.


In [2]:
from db import connect
from explore import analyze, database_hash, print_table
before = database_hash(DATABASE)
with connect(DATABASE, read_only=True) as connection:
    RESULT = analyze(connection, PARAMS, ROOT)
assert database_hash(DATABASE) == before, 'Database bytes changed'
print('Read-only analysis completed; database SHA-256 unchanged:', before)
print('Full catalog:', RESULT['catalog_n'], 'Selected cohort:', RESULT['cohort_n'])
print('Duplicate-field and identity checks:', RESULT['audit'])


Read-only analysis completed; database SHA-256 unchanged: a47682e00aec4c38df7a46739a1bd21a5585920c2a27405d0862cdda370bcb83
Full catalog: 14877 Selected cohort: 1625
Duplicate-field and identity checks: {'duplicate_analytical_fields': 0, 'identity_mappings': 0}


## 2.1 Raw versus cleaned

Question: how much does trusting SQLite's raw price cast change the mean EUR catalog price by brand? The M2 baseline removes the `About`/`EUR` wrapper and casts the remaining source string without repairing grouped thousands; this is explicitly a naive raw baseline. Cleaned results use the M3 parser and primary flags. Both queries and cohort differences are retained in the report. Neither result establishes MSRP.


In [3]:
for name in ['m4_raw.sql', 'm4_cohort.sql']:
    print(name)
    print((ROOT / 'sql/queries' / name).read_text())
print('M2 raw baseline:')
print_table(RESULT['raw'], ['brand', 'catalog_n', 'n', 'mean_price'])
print('M3 cleaned:')
print_table(RESULT['groups'], ['brand', 'catalog_n', 'price_n', 'mean', 'median', 'p95'])


m4_raw.sql
-- M2-only baseline: latest rows, raw Android prefix, raw year cast, no flags.
-- Deliberately naive SQLite CAST of About ... EUR; no cleaning or variants.
WITH models AS (
 SELECT p.phone_key,substr(p.name,1,instr(p.name || ' ',' ')-1) AS brand,
 MAX(CASE WHEN s.source_field='os' THEN s.value_text END) AS os,
 MAX(CASE WHEN s.source_field='year' THEN CAST(s.value_text AS INTEGER) END) AS year,
 MAX(CASE WHEN s.source_field='price' AND s.value_text LIKE :price_pattern
 THEN CAST(replace(replace(s.value_text,'About ',''),' EUR','') AS REAL) END) AS price
 FROM phone p JOIN specification s ON s.phone_key=p.phone_key AND s.retrieved_at=p.retrieved_at
 GROUP BY p.phone_key,p.name
)
SELECT brand,COUNT(*) AS catalog_n,COUNT(price) AS n,AVG(price) AS mean_price
FROM models WHERE os LIKE :os_pattern AND year BETWEEN :start AND :end
 AND brand IN (SELECT value FROM json_each(:brands)) GROUP BY brand ORDER BY brand;

m4_cohort.sql
-- One row per current canonical catalog model. Every 

## 2.2 The question and summary

Question: among the four selected Android brand cohorts, how do typical EUR catalog amounts differ, and how does RAM relate to price where a single configuration is available? The median answers the typical-model question; the mean answers average amount per priced model. P95 describes the expensive tail. Every statistic carries its usable-price count, with catalog counts as the coverage denominator. Brand-wide descriptive aggregates need catalog n >= 20, usable price n >= 10, and >= 50% price coverage; no inference is made for a failed group.


In [4]:
print_table(RESULT['groups'], ['brand','catalog_n','price_n','mean','median','p95','eligible'])
for statistic in ['median', 'mean', 'p95']:
    print(statistic, 'descending:', [r['brand'] for r in sorted(RESULT['groups'], key=lambda r:r[statistic], reverse=True)])


| brand | catalog_n | price_n | mean | median | p95 | eligible |
|---|---|---|---|---|---|---|
| Huawei | 303 | 298 | 283.054 | 200.000 | 764.500 | yes |
| Samsung | 715 | 624 | 266.177 | 200.000 | 658.500 | yes |
| Sony | 156 | 145 | 256.455 | 210.000 | 542.000 | yes |
| Xiaomi | 451 | 375 | 283.154 | 220.000 | 673.000 | yes |
median descending: ['Xiaomi', 'Sony', 'Huawei', 'Samsung']
mean descending: ['Xiaomi', 'Huawei', 'Samsung', 'Sony']
p95 descending: ['Huawei', 'Xiaomi', 'Samsung', 'Sony']


## 2.3 Shape

Question: what features of the distribution would a single summary hide? Three natural EUR bin widths are compared below; EUR 100 is used in the shared-scale histogram. The ECDF removes dependence on bin placement. All priced rows are included for descriptive shape, including thin yearly cells; the figures do not turn those cells into yearly estimates.

![Price histograms](../figures/01_price_histogram.svg)

![Price ECDF](../figures/02_price_ecdf.svg)


In [5]:
print_table(RESULT['bins'], ['width','nonempty_bins','peak_bin_n','upper','n'])
print('Overall priced cohort:', RESULT['overall'])


| width | nonempty_bins | peak_bin_n | upper | n |
|---|---|---|---|---|
| 50 | 37 | 270 | 2,900.000 | 1442 |
| 100 | 25 | 537 | 2,900.000 | 1442 |
| 200 | 14 | 645 | 2,900.000 | 1442 |
Overall priced cohort: {'n': 1442, 'mean': 273.10226768377254, 'median': 210.0, 'p95': 670.0, 'min': 50.0, 'max': 2900.0}


## 2.4 Coverage and flag sensitivity

Question: which brand-years have enough observed prices to support a summary? The denominator is the landed catalog cohort, not the unknown universe of every released phone. One entity has one latest observation, so its price availability is 0/1; the annual and brand tables sum those indicators. No observation is invented for an absent field. A table with failed groups is retained for auditing, but failed groups do not enter the annual plotted comparison. The 50% coverage and minimum-count rules are pragmatic stability screens, not a guarantee against selection bias. Compare the same equal-year contrast with and without this rule. The strict filter removes unusual values, including legitimate premium devices.

![Coverage by brand and year](../figures/03_coverage.svg)


In [6]:
print_table(RESULT['groups'], ['brand','catalog_n','price_n','joint_n','price_coverage','ram_price_coverage'])
print_table(RESULT['annual'], ['brand','year','catalog_n','price_n','joint_n','price_coverage','eligible'])
print('Primary flags:', PARAMS['flags'])
print_table(RESULT['groups'], ['brand','price_n','mean','median','p95'])
print('Strict flags:', PARAMS['strict_flags'])
print_table(RESULT['strict_groups'], ['brand','price_n','mean','median','p95'])
print('With coverage rule:', RESULT['comparison']['estimate'], RESULT['comparison']['years'])
print('Without coverage rule:', RESULT['without_coverage_rule'])


| brand | catalog_n | price_n | joint_n | price_coverage | ram_price_coverage |
|---|---|---|---|---|---|
| Huawei | 303 | 298 | 136 | 0.983 | 0.449 |
| Samsung | 715 | 624 | 276 | 0.873 | 0.386 |
| Sony | 156 | 145 | 110 | 0.929 | 0.705 |
| Xiaomi | 451 | 375 | 38 | 0.831 | 0.084 |
| brand | year | catalog_n | price_n | joint_n | price_coverage | eligible |
|---|---|---|---|---|---|---|
| Huawei | 2010 | 8 | 8 | 4 | 1.000 | no |
| Huawei | 2011 | 15 | 15 | 12 | 1.000 | no |
| Huawei | 2012 | 25 | 25 | 19 | 1.000 | yes |
| Huawei | 2013 | 22 | 22 | 19 | 1.000 | yes |
| Huawei | 2014 | 27 | 26 | 21 | 0.963 | yes |
| Huawei | 2015 | 23 | 23 | 14 | 1.000 | yes |
| Huawei | 2016 | 21 | 21 | 7 | 1.000 | yes |
| Huawei | 2017 | 26 | 26 | 12 | 1.000 | yes |
| Huawei | 2018 | 32 | 32 | 6 | 1.000 | yes |
| Huawei | 2019 | 41 | 39 | 10 | 0.951 | yes |
| Huawei | 2020 | 46 | 46 | 10 | 1.000 | yes |
| Huawei | 2021 | 9 | 8 | 2 | 0.889 | no |
| Huawei | 2022 | 4 | 3 | 0 | 0.750 | no |
| Huawei | 20

## 2.5 Announcement cohorts and collection clock

Question: how do annual and five-year device cohorts differ? Announcement year is a source calendar year with no recoverable timezone; it is not a UTC timestamp. It must not be tz-converted or interpreted as a transaction date. Retrieval is explicitly UTC and only documents the collection window. The 2026 announcement cohort is excluded because the collection predates year end. The 2010 left edge and 2025 right edge are inspected in the coverage table; passing a rule does not establish completeness. Annual and five-year medians are cohort patterns, not historical price trajectories. Day-of-week, holidays, and seasonality cannot be tested from year-only dates. Brand within announcement cohort is the alternative grouping structure available here.

![Announcement cohort summaries](../figures/04_announcement_cohorts.svg)


In [7]:
print('Retrieval UTC range:', RESULT['retrieval_range'])
print('Announcement range retained:', RESULT['cohort_year_range'])
print('Boundary brand-years:')
print_table([r for r in RESULT['annual'] if r['year'] in [PARAMS['start_year'],PARAMS['end_year']]], ['brand','year','catalog_n','price_n','price_coverage','eligible'])
print('Five-year cohorts; same flag and coverage rules:')
print_table(RESULT['periods'], ['brand','period','catalog_n','price_n','median','eligible'])


Retrieval UTC range: ['2026-10-09T13:12:26.109302Z', '2026-10-11T00:13:23.874785Z']
Announcement range retained: [2010, 2025]
Boundary brand-years:
| brand | year | catalog_n | price_n | price_coverage | eligible |
|---|---|---|---|---|---|
| Huawei | 2010 | 8 | 8 | 1.000 | no |
| Huawei | 2025 | 3 | 3 | 1.000 | no |
| Samsung | 2010 | 27 | 27 | 1.000 | yes |
| Samsung | 2025 | 39 | 13 | 0.333 | no |
| Sony | 2010 | 4 | 4 | 1.000 | no |
| Sony | 2025 | 2 | 1 | 0.500 | no |
| Xiaomi | 2025 | 62 | 27 | 0.435 | no |
Five-year cohorts; same flag and coverage rules:
| brand | period | catalog_n | price_n | median | eligible |
|---|---|---|---|---|---|
| Huawei | 2010 | 97 | 96 | 140.000 | yes |
| Huawei | 2015 | 143 | 141 | 230.000 | yes |
| Huawei | 2020 | 60 | 58 | 255.000 | yes |
| Huawei | 2025 | 3 | 3 | 240.000 | no |
| Samsung | 2010 | 299 | 296 | 180.000 | yes |
| Samsung | 2015 | 183 | 181 | 220.000 | yes |
| Samsung | 2020 | 194 | 134 | 210.000 | yes |
| Samsung | 2025 | 39 | 13 | 

## 2.6 Relationships and drivers

Question: how much of RAM versus price is shared generation/brand structure or a few extremes? Inspect the scatter first. Both coefficients use the same complete pairs; at least 10 pairs are required per brand or brand-year coefficient. Coefficients on all pairs are descriptive complete-case results, with their denominator against the eligible catalog shown above. Sensitivities remove rows above the marginal 99th percentiles, omit one brand at a time, and demean RAM and price within brand-year cells with >= 10 pairs. Spearman on those residuals is a sensitivity, not a formal partial rank coefficient. No missing configuration is parsed or chosen in this notebook.

Device type is a second candidate driver: Android also covers tablets, and the cleaned layer has no device class. Display size is the measured attribute closest to it, so the coefficients are recomputed separately for phone-sized screens and for screens of at least `large_display_in` inches (7.0, the smallest tablet screen among these pairs). The large-screen pairs are listed in full, because a screen-size group is not a device class and nothing here relabels a model.

![RAM and price by brand](../figures/05_ram_price.svg)


In [8]:
print('Overall:', RESULT['relationship'])
print('Trim cutoffs:', RESULT['trim_thresholds'], 'removed:', RESULT['trim_removed'])
print('After trim:', RESULT['trimmed'])
print('Within brand:', json.dumps(RESULT['by_brand'], indent=2))
print('Leave one brand out:', json.dumps(RESULT['leave_one_brand_out'], indent=2))
print_table(RESULT['within_brand_year'], ['brand','year','n','pearson','spearman'])
print('Brand-year demeaned:', RESULT['group_demeaned'])
screens = RESULT['display_split']
print(f"Display split at {screens['threshold_in']} in; pairs without a display size: {screens['display_missing']}")
print_table([dict(group=f"display < {screens['threshold_in']} in", **screens['phone_sized']),
             dict(group=f"display >= {screens['threshold_in']} in", **screens['large_screen'])],
            ['group', 'n', 'pearson', 'spearman'])
print('Large-screen pairs by brand:', screens['large_screen_by_brand'])
print('Large-screen pairs, most expensive first:')
print_table(RESULT['large_screen_pairs'], ['name', 'year', 'display_in', 'ram', 'price'])


Overall: {'n': 560, 'pearson': 0.673614759441416, 'spearman': 0.5955581762251693}
Trim cutoffs: {'ram_gb': 12.0, 'price_eur': 1720.4999999999984} removed: 6
After trim: {'n': 554, 'pearson': 0.6237102810853171, 'spearman': 0.5823354952277688}
Within brand: {
  "Samsung": {
    "n": 276,
    "pearson": 0.7916590883316404,
    "spearman": 0.612265967389255
  },
  "Xiaomi": {
    "n": 38,
    "pearson": 0.5205084421866268,
    "spearman": 0.7185534737505554
  },
  "Huawei": {
    "n": 136,
    "pearson": 0.628297427635333,
    "spearman": 0.6771499519448825
  },
  "Sony": {
    "n": 110,
    "pearson": 0.7755069813057494,
    "spearman": 0.6287397719569316
  }
}
Leave one brand out: {
  "Samsung": {
    "n": 284,
    "pearson": 0.6027452831553274,
    "spearman": 0.5946384018203513
  },
  "Xiaomi": {
    "n": 522,
    "pearson": 0.7310437522257066,
    "spearman": 0.6265989420034423
  },
  "Huawei": {
    "n": 424,
    "pearson": 0.6942827802854563,
    "spearman": 0.5684173878423082
  },

## 2.7 A comparison, an interval and a look count

Question: how much larger is Samsung's typical catalog amount than Xiaomi's in common supported years? The primary estimand is the equal-year mean of within-year median differences, with paired two-year circular block bootstrap resampling. This keeps the brands paired and preserves some short-range dependence; only a small number of years are available. The pooled median gap answers a different, mixture-weighted question. We inspect all six pooled brand pairs, then the preselected Samsung/Xiaomi contrast, its annual strata, coverage sensitivity, and flag sensitivity. These are exploratory looks on the same extract, with no multiplicity-adjusted significance claim or held-out test.

![Brand comparison](../figures/06_brand_comparison.svg)


In [9]:
print('Comparison:', json.dumps(RESULT['comparison'], indent=2))
print('All six pooled brand-pair looks:')
print_table(RESULT['all_pairs'], ['a','b','pooled_median_difference','a_n','b_n'])


Comparison: {
  "a": "Samsung",
  "b": "Xiaomi",
  "years": [
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ],
  "paired": [
    {
      "year": 2018,
      "difference": -60.0,
      "a_n": 28,
      "b_n": 23,
      "a_median": 210.0,
      "b_median": 270.0
    },
    {
      "year": 2019,
      "difference": 0.0,
      "a_n": 44,
      "b_n": 34,
      "a_median": 260.0,
      "b_median": 260.0
    },
    {
      "year": 2020,
      "difference": 105.0,
      "a_n": 44,
      "b_n": 48,
      "a_median": 275.0,
      "b_median": 170.0
    },
    {
      "year": 2021,
      "difference": -55.0,
      "a_n": 31,
      "b_n": 50,
      "a_median": 160.0,
      "b_median": 215.0
    },
    {
      "year": 2022,
      "difference": -60.0,
      "a_n": 26,
      "b_n": 66,
      "a_median": 185.0,
      "b_median": 245.0
    },
    {
      "year": 2023,
      "difference": -10.0,
      "a_n": 17,
      "b_n": 35,
      "a_median": 210.0,
      "b_median": 220.0
 

## 2.8 Findings and decisions

The report is written from these executed results. Attribution is explicitly pending member review; preparation does not establish who explored or can explain a finding. There is no held-out confirmation of the price/specification relationships. The cleaning register is updated with interpretation/selection limits rather than an unsupported new parser fix.


In [10]:
from explore_report import write_report
write_report(RESULT, ROOT)
evidence = {'database_sha256_before':before, 'database_sha256_after':database_hash(DATABASE),
            'database_unchanged':before==database_hash(DATABASE),
            'figure_files':sorted(p.name for p in (ROOT/'figures').glob('*.svg')),
            'parameters':PARAMS}
(ROOT/'docs/m4').mkdir(parents=True, exist_ok=True)
(ROOT/'docs/m4/results.json').write_text(json.dumps(RESULT,indent=2)+'\n',encoding='utf-8')
(ROOT/'docs/m4/notebook-verification.json').write_text(json.dumps(evidence,indent=2)+'\n',encoding='utf-8')
print('Wrote report, six SVG figures, aggregate results and read-only verification.')
print('Parameters actually used:', json.dumps(PARAMS, sort_keys=True))


Wrote docs\eda_report.md
Wrote report, six SVG figures, aggregate results and read-only verification.
Parameters actually used: {"bin_trials": [50, 100, 200], "bin_width": 100, "block_length": 2, "bootstrap_reps": 5000, "brands": ["Samsung", "Xiaomi", "Huawei", "Sony"], "comparison": ["Samsung", "Xiaomi"], "comparison_start": 2015, "currency": "EUR", "end_year": 2025, "flags": ["ok", "unit_converted", "unusual_low", "unusual_high"], "large_display_in": 7.0, "min_coverage": 0.5, "min_joint": 10, "min_models": 20, "min_prices": 10, "os": "Android", "seed": 20261111, "start_year": 2010, "strict_flags": ["ok", "unit_converted"]}
